In [2]:
!pip install yfinance pandas

In [4]:
import yfinance as yf

# Fetch Zomato ticker data from the National Stock Exchange of India
zomato = yf.Ticker("ZOMATO.NS")

# Pull annual financials and swap rows/columns so it reads easily
df = zomato.financials.T

# Keep only the rows we need to see
available_columns = [col for col in ['Total Revenue', 'Net Income From Continuing Operation Net Minority Interest'] if col in df.columns]
summary = df[available_columns]

print(summary)


Empty DataFrame
Columns: []
Index: []


In [5]:
import pandas as pd

print("--- INITIALIZING ZOMATO (ETERNAL LTD) PIPELINE ---")

# Real historical revenues in ₹ Crores (Cr)
# FY23: 7,079 Cr | FY24: 12,114 Cr | FY25: 21,581 Cr
years = ['2023A', '2024A', '2025A', '2026E', '2027E', '2028E']
revenue = [7079, 12114, 21581, 0, 0, 0] # 2026-2028 will be calculated below

# Set an aggressive but realistic 35% growth rate for Zomato's Quick Commerce (Blinkit) expansion
growth_rate = 0.35

# Projecting the next 3 years automatically
for i in range(3, 6):
    revenue[i] = round(revenue[i-1] * (1 + growth_rate))

# Create a clean DataFrame
dcf_df = pd.DataFrame({
    'Fiscal Year': years,
    'Revenue (₹ Cr)': revenue
})

print(dcf_df.to_string(index=False))


--- INITIALIZING ZOMATO (ETERNAL LTD) PIPELINE ---
Fiscal Year  Revenue (₹ Cr)
      2023A            7079
      2024A           12114
      2025A           21581
      2026E           29134
      2027E           39331
      2028E           53097


In [6]:
# Create lists for the upcoming calculations
ebit = []
taxes = []
capex_net = []
fcff = []

# Constants based on current Indian market and tech sector baselines
ebit_margin = 0.08    # 8% Operating margin
tax_rate = 0.25       # 25% Corporate tax
reinvestment_rate = 0.02 # 2% Net Capital Reinvestment

# Calculate metrics for the projected years (2026E, 2027E, 2028E)
projected_revenues = [29134, 39331, 53097]

for rev in projected_revenues:
    val_ebit = round(rev * ebit_margin)
    val_tax = round(val_ebit * tax_rate)
    val_reinvest = round(rev * reinvestment_rate)
    val_fcff = val_ebit - val_tax - val_reinvest

    ebit.append(val_ebit)
    taxes.append(val_tax)
    capex_net.append(val_reinvest)
    fcff.append(val_fcff)

# Display the cash flow pipeline
fcff_df = pd.DataFrame({
    'Fiscal Year': ['2026E', '2027E', '2028E'],
    'Revenue (₹ Cr)': projected_revenues,
    'Operating Profit (EBIT)': ebit,
    'Expected Taxes': taxes,
    'Net Reinvestment': capex_net,
    'Free Cash Flow (FCFF)': fcff
})

print("--- ZOMATO PROJECTED FREE CASH FLOWS ---")
print(fcff_df.to_string(index=False))


--- ZOMATO PROJECTED FREE CASH FLOWS ---
Fiscal Year  Revenue (₹ Cr)  Operating Profit (EBIT)  Expected Taxes  Net Reinvestment  Free Cash Flow (FCFF)
      2026E           29134                     2331             583               583                   1165
      2027E           39331                     3146             786               787                   1573
      2028E           53097                     4248            1062              1062                   2124


In [7]:
wacc = 0.12  # 12% Cost of Capital for Indian Equities
g = 0.05     # 5% Long-term terminal growth rate
outstanding_shares = 893  # Zomato has ~893 Crore shares outstanding

# 1. Discount the individual cash flows (2026E, 2027E, 2028E) back to Present Value (PV)
pv_2026 = 1165 / ((1 + wacc) ** 1)
pv_2027 = 1573 / ((1 + wacc) ** 2)
pv_2028 = 2124 / ((1 + wacc) ** 3)
sum_pv_fcff = pv_2026 + pv_2027 + pv_2028

# 2. Calculate Terminal Value at the end of 2028 using the perpetuity formula
terminal_value = (2124 * (1 + g)) / (wacc - g)
pv_terminal_value = terminal_value / ((1 + wacc) ** 3)

# 3. Calculate Total Intrinsic Value of the Enterprise
enterprise_value = sum_pv_fcff + pv_terminal_value

# 4. Derive Intrinsic Price Per Share
intrinsic_value_per_share = enterprise_value / outstanding_shares

print("--- DCF VALUATION SUMMARY ---")
print(f"Present Value of 3-Year Cash Flows: ₹ {round(sum_pv_fcff)} Cr")
print(f"Present Value of Terminal Value   : ₹ {round(pv_terminal_value)} Cr")
print(f"Total Intrinsic Enterprise Value  : ₹ {round(enterprise_value)} Cr")
print(f"CALCULATED TARGET PRICE PER SHARE : ₹ {round(intrinsic_value_per_share, 2)}")



--- DCF VALUATION SUMMARY ---
Present Value of 3-Year Cash Flows: ₹ 3806 Cr
Present Value of Terminal Value   : ₹ 22677 Cr
Total Intrinsic Enterprise Value  : ₹ 26483 Cr
CALCULATED TARGET PRICE PER SHARE : ₹ 29.66


In [9]:
# ==========================================
# STAGE 4: WEALTH MANAGEMENT PORTFOLIO (AGGRESSIVE PROFILE)
# ==========================================
print("\n=== STAGE 4: AGGRESSIVE WEALTH MANAGEMENT CLIENT ALLOCATION ===")
total_portfolio = 100000000  # ₹10 Crore Core Asset Pool

allocations = {
    'Zomato Equity (High Growth Tech)': 0.25,      # Increased to 25% (High Alpha)
    'Nifty 50 Index (Core Equity Anchor)': 0.45,   # Increased to 45% (Market Capture)
    'Sovereign Gold Bonds (Alternative Hedge)': 0.10, # Reduced to 10%
    'GOI Bonds (Capital Protection Floor)': 0.15,  # Reduced to 15% (Risk-on stance)
    'Liquid Cash / Money Market Funds': 0.05       # Reduced to 5% (Minimized drag)
}

portfolio_rows = []
for asset, weight in allocations.items():
    amount = total_portfolio * weight
    portfolio_rows.append({'Asset Class': asset, 'Weight (%)': weight * 100, 'Amount (INR)': amount})
    print(f"{asset:<40} | Weight: {weight*100:>3.0f}% | Value: ₹ {amount:,.0f}")

# ==========================================
# STAGE 5: AUTO-EXPORT FILE FOR TABLEAU
# ==========================================
portfolio_df = pd.DataFrame(portfolio_rows)
portfolio_df.to_csv('wealth_portfolio_aggressive.csv', index=False)
print("\n[SUCCESS] 'wealth_portfolio_aggressive.csv' generated. Ready for Tableau upload.")



=== STAGE 4: AGGRESSIVE WEALTH MANAGEMENT CLIENT ALLOCATION ===
Zomato Equity (High Growth Tech)         | Weight:  25% | Value: ₹ 25,000,000
Nifty 50 Index (Core Equity Anchor)      | Weight:  45% | Value: ₹ 45,000,000
Sovereign Gold Bonds (Alternative Hedge) | Weight:  10% | Value: ₹ 10,000,000
GOI Bonds (Capital Protection Floor)     | Weight:  15% | Value: ₹ 15,000,000
Liquid Cash / Money Market Funds         | Weight:   5% | Value: ₹ 5,000,000

[SUCCESS] 'wealth_portfolio_aggressive.csv' generated. Ready for Tableau upload.


In [10]:
# =========================================================================
# RE-RUN STAGE 4: ALIGNING LABELS TO ADVANCED INSTITUTIONAL INSTRUMENTS
# =========================================================================
print("\n=== STAGE 4: ADVANCED INSTITUTIONAL WEALTH ALLOCATION ===")
total_portfolio = 100000000

allocations = {
    'Zomato Direct Equity (Target Asset)': 0.25,
    'Nifty Midcap 150 Index Fund (Alpha Core)': 0.35,
    'Pre-IPO / Venture Capital (SEBI Cat II AIF)': 0.15,
    'Equity-Linked Debentures / Derivatives': 0.20,
    'Liquid Arbitrage / Tactical Cash Buffer': 0.05
}

portfolio_rows = []
for asset, weight in allocations.items():
    amount = total_portfolio * weight
    portfolio_rows.append({'Asset Class': asset, 'Weight (%)': weight * 100, 'Amount (INR)': amount})
    print(f"{asset:<45} | Weight: {weight*100:>3.0f}% | Value: ₹ {amount:,.0f}")

portfolio_df = pd.DataFrame(portfolio_rows)
portfolio_df.to_csv('advanced_wealth_portfolio.csv', index=False)



=== STAGE 4: ADVANCED INSTITUTIONAL WEALTH ALLOCATION ===
Zomato Direct Equity (Target Asset)           | Weight:  25% | Value: ₹ 25,000,000
Nifty Midcap 150 Index Fund (Alpha Core)      | Weight:  35% | Value: ₹ 35,000,000
Pre-IPO / Venture Capital (SEBI Cat II AIF)   | Weight:  15% | Value: ₹ 15,000,000
Equity-Linked Debentures / Derivatives        | Weight:  20% | Value: ₹ 20,000,000
Liquid Arbitrage / Tactical Cash Buffer       | Weight:   5% | Value: ₹ 5,000,000


In [14]:
import pandas as pd

# Define the complete institutional-grade portfolio data structure
data = {
    'Asset Class': [
        'Zomato Direct Equity',
        'Nifty Midcap 150 Index Fund',
        'Pre-IPO / Venture Capital (SEBI Cat II AIF)',
        'Equity-Linked Debentures',
        'Liquid Arbitrage Cash'
    ],
    'Allocation (INR)': [25000000, 35000000, 15000000, 20000000, 5000000],
    'Weight (%)': [25, 35, 15, 20, 5]
}

# Convert dictionary to a DataFrame and export to CSV
df = pd.DataFrame(data)
df.to_csv('advanced_wealth_portfolio.csv', index=False)

# Trigger automatic browser download
from google.colab import files
files.download('advanced_wealth_portfolio.csv')

print("[SUCCESS] Data pipeline executed. File 'advanced_wealth_portfolio.csv' is downloading.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[SUCCESS] Data pipeline executed. File 'advanced_wealth_portfolio.csv' is downloading.


In [1]:
# Live Interactive Dashboard: [https://public.tableau.com/views/AssetAllocationFramework/Sheet2?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link]

